# Does the detector fusion beat the current pipeline? The holdout verdict (CPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. **Run it once with `PILOT = True` first** (6 holdout files, the ones a pilot export has): it checks the plumbing, and its numbers are not a result.

This is the test of phase F of `docs/ROADMAP_V2.md`: the detector says which pixels go together and how sure it is, the U-Net says which pixels are filament, and `filament.detector_fusion` joins them. This notebook scores that fusion on the date holdout, one axis at a time, without a GPU and without running the U-Net again: the U-Net side is the probability cache of the Kaggle notebook `export-probs-r34`, the detector side is the output of `notebooks/export_detections.ipynb`.

**The base to beat** is the current pipeline: flip-TTA r34, threshold 0.45, close_size 11, min_area 400, object filter mean >= 0.8. Pooled PQ 0.4255 in sample, 0.4210 cross-fitted (`docs/RESULTS.md`). The notebook recomputes it from the cache and stops if it differs from 0.4255 by more than 0.002.

**Axes, one at a time** from the defaults of `filament.detector_fusion` (dilation 5, `emit_unclaimed` off, no cut), on the same holdout through `FusionConfig`-based claims:

1. score cut on the raw detector score (0.05 .. 0.7),
2. the same cut on the cross-fitted **calibrated** score (an isotonic fit by date, out of fold),
3. dilation of the detector mask before it claims U-Net pixels (0 .. 15 px),
4. `emit_unclaimed` on and off.

The grids are untuned placeholders chosen before any result. Picking the best row would overstate the gain, so the verdict is a **cross-fit over all of them by date on 5 partitions** (`filament.stats`, rule of `filament.axis_report`: a gain of at least 0.005 in at least 90% of paired-bootstrap resamples, in at least 4 of the 5 partitions), plus the paired bootstrap of every row against the base. The detector-only PQ (no U-Net) is printed as a sanity row. The notebook also logs how many detections the deduplication drops, and how many of those are cross-window pairs that the merge had left apart.

> **Rule for a submission.** A submission without Nacho's OK needs all of these: a `diff` above **+0.01** on the `VERDICT` line, `adopted` of at least 4 of the 5 partitions (the `adopted=k/5` field of that line), and a run that is not a pilot. With +0.01 or less, or fewer partitions adopted, the submission needs his OK (`AGENTS.md`).
>
> **What `diff` is against.** The base at its fixed setting, in sample (pooled PQ 0.4255), not the cross-fitted base (0.4210). The comparison is stricter on purpose: both arms share the same frozen U-Net mask, so the selection bias of the threshold and the filter is in both. The `VERDICT` line prints that base as `base_fixed_in_sample`. Its `config` is the setting chosen most often, while `pq` and `diff` mix the settings chosen per fold, so a submission with that one config does not have exactly that number.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code`, a version that contains `filament.detector_sweep`.
3. The output of the Kaggle notebook `ignacioaguirrecandia/export-probs-r34` (its cache for `MODEL = "r34"`).
4. The output of `export_detections.ipynb` (`detections_holdout.jsonl.gz` and `detector_export_manifest.json`).

**Settings:** Accelerator **None** (no GPU), Internet **Off** is enough. **Output:** `/kaggle/working/holdout_detector_sweep.csv`, small and with no mask data, so it can go in `results/`. The last printed line is `VERDICT <config> pq=<crossfit> diff=<vs base> fraction_better=<f> adopted=<k>/5 base_fixed_in_sample=<pq>`. The running time is not measured yet [VERIFY]; a progress line is printed every 10 frames.

It reads only the holdout side of the cache and the holdout detections. It does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle.

## 1. `src` on the path

`src` is the directory that contains `filament/detector_sweep.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match.

In [ ]:
import os
import sys
from pathlib import Path

MARKER = Path("filament") / "detector_sweep.py"


def _local_src() -> Path | None:
    """A checkout that already has the module this notebook needs (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / MARKER).is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    The marker may sit at any depth. Directories named ``competitions`` are not
    walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / MARKER).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the private
    dataset's ``src`` is used in place. It is not copied to ``/kaggle/working``: a
    copy there would be saved with this notebook's output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            f"Could not find src/{MARKER.as_posix()} in the working directory or "
            "under /kaggle/input (any depth outside competitions). Attach the "
            "dataset ignacioaguirrecandia/solar-code. If it is attached, its "
            "version predates the detector modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings, the cache and the detections

- `BASE` is the current setting (`filament.gap_analysis.CURRENT_POINT`) and `ADOPTED_OBJECT_FILTER` its object filter. Every difference is measured against this pipeline.
- The cache has to come from exactly the checkpoint `MODEL` names: its manifest goes through the same guard as a weights file. Leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`.
- The detection export must have been made with the window geometry of `frame_windows()`. Its manifest is checked.

In [ ]:
# ---- Settings. Nothing below this cell needs editing. -----------------------
PILOT = False  # True: PILOT_FILES holdout files, to check the plumbing; not a result
PILOT_FILES = 6  # the frames of a pilot export (export_detections.ipynb, PILOT_FRAMES)
MODEL = "r34"
ALLOW_UNVERIFIED_WEIGHTS = False
EXPECTED_BASE_PQ = 0.4255  # pooled PQ in sample of the base (docs/RESULTS.md)
BASE_PQ_TOLERANCE = 0.002

In [ ]:
import json
import time

import numpy as np

from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    annotator_views,
    load_train_annotations,
)
from filament.axis_report import (
    MIN_PARTITIONS_ADOPTED,
    axis_rows,
    crossfit_rows,
    format_axis_rows,
    write_axis_rows,
)
from filament.baseline import on_disk
from filament.checkpoint import crop_checkpoint, verify_and_report
from filament.components import (
    ADOPTED_OBJECT_FILTER,
    filter_components,
    object_cut_from_config,
)
from filament.crop_data import read_gray
from filament.detector_data import build_detector_splits, frame_windows
from filament.detector_export import (
    EXPORT_MANIFEST,
    detections_name,
    find_detection_export,
    read_detections,
    window_rects,
)
from filament.detector_fusion import DEFAULT_CONFIG
from filament.detector_sweep import (
    SUBMIT_GAIN,
    DedupStats,
    axis_settings,
    calibrate_crossfit,
    claim_file,
    counts_for,
    crossfit_summary,
    submit_without_asking,
    unique_settings,
    verdict_line,
)
from filament.detector_train import subsample_evenly
from filament.gap_analysis import CURRENT_POINT, holdout_dates
from filament.instances import labels_for_min_areas
from filament.metrics import PqCounts, fast_view_counts
from filament.probcache import (
    ProbCache,
    find_cache_root,
    manifest_fingerprint,
    read_manifest,
    split_dir,
)
from filament.stats import weighted_pq
from filament.sweep import view_masks

EXPECTED = crop_checkpoint(MODEL)
BASE = CURRENT_POINT
OBJECT_CUT = object_cut_from_config(ADOPTED_OBJECT_FILTER)
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "detector_sweep"
)
OUTPUT_PATH = OUTPUT_DIR / "holdout_detector_sweep.csv"

CACHE_ROOT = find_cache_root(MODEL, INPUT_ROOT)
print(f"cache={CACHE_ROOT}")
verify_and_report(
    manifest_fingerprint(read_manifest(CACHE_ROOT, MODEL)),
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
)

EXPORT_DIR = find_detection_export(INPUT_ROOT, "holdout")
export_manifest = json.loads((EXPORT_DIR / EXPORT_MANIFEST).read_text(encoding="utf-8"))
print(
    f"detections={EXPORT_DIR} checkpoint={export_manifest['checkpoint_file']} "
    f"sha256={export_manifest['checkpoint_sha256']} resolution={export_manifest['resolution']} "
    f"threshold={export_manifest['score_threshold']} pilot={export_manifest['pilot']}"
)
if export_manifest["pilot"] and not PILOT:
    raise RuntimeError(
        "These detections came from a pilot export (a few frames). Export for real first."
    )
WINDOWS = frame_windows()
if export_manifest["windows"] != [[w.x0, w.y0, w.x1, w.y1] for w in WINDOWS]:
    raise RuntimeError(
        "The export used other windows than frame_windows(); the geometry would be wrong."
    )
RECTS = window_rects(WINDOWS)

## 3. Date holdout

The same split as the U-Net notebooks and the detector: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Every holdout file has to be in the cache and in the detection file.

In [ ]:
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
holdout = build_detector_splits(loaded).holdout
views_total = sum(len(annotator_views(image)) for image in holdout)
print(f"holdout files={len(holdout)} annotator views={views_total}")
if not PILOT and (len(holdout), views_total) != (142, 231):
    print(
        "WARNING: the holdout is not 142 files with 231 views. [VERIFY] the data version."
    )
if PILOT:
    holdout = list(subsample_evenly(holdout, PILOT_FILES))
    print(f"PILOT: {len(holdout)} files; these numbers are not a result")
cache = ProbCache(split_dir(CACHE_ROOT, MODEL, "holdout"))
missing = [image.file_name for image in holdout if image.file_name not in cache]
if missing:
    raise RuntimeError(
        f"{len(missing)} holdout files are not in the cache, for example {missing[0]}."
    )

wanted = {image.file_name for image in holdout}
detections_by_file = {}
for file_name, shape, detections in read_detections(
    EXPORT_DIR / detections_name("holdout")
):
    if file_name in wanted:
        detections_by_file[file_name] = detections
absent = sorted(wanted - set(detections_by_file))
if absent:
    raise RuntimeError(
        f"{len(absent)} holdout files have no entry in the detection file, e.g. {absent[0]}."
    )
print(
    f"detections read: {sum(len(v) for v in detections_by_file.values())} in {len(detections_by_file)} frames"
)

## 4. Score every setting from one claim per frame

For each frame: the base pipeline's mask (threshold, closing, `min_area`, solar-disk gate, object filter), the ground-truth masks of every annotator view, then `claim_file`: merge across windows and deduplicate once, claim the U-Net pixels once per `(dilation, emit_unclaimed)` with the same `on_disk` gate as the pipeline, and record the matches. A score cut then needs no pixels: `counts_for` is `selection_counts` on the recorded matches. Calibration needs every frame first, so it follows the loop.

In [ ]:
AXES = axis_settings()
SETTINGS = unique_settings(AXES)
CLAIM_CONFIGS = list(dict.fromkeys(setting.config for setting in SETTINGS))
print(
    f"{len(SETTINGS)} settings on {len(AXES)} axes, {len(CLAIM_CONFIGS)} claims per frame"
)


def base_labels(gray, probability, disk):
    # The pipeline of docs/PROJECT_REPORT.md: threshold, closing, min_area, disk gate, object filter.
    semantic = (probability >= BASE.threshold).astype(np.uint8)
    labels = labels_for_min_areas(
        semantic, gray, [BASE.min_area], close_size=BASE.close_size, disk=disk
    )[BASE.min_area]
    return filter_components(labels, probability, OBJECT_CUT)


base_counts = []
claimed = []
begin = time.monotonic()
for index, image in enumerate(holdout, start=1):
    gray = read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))
    probability = cache.mean(image.file_name)
    if probability.shape != gray.shape:
        raise ValueError(
            f"{image.file_name}: probability {probability.shape} vs frame {gray.shape}"
        )
    disk = on_disk(gray)
    kept = base_labels(gray, probability, disk)
    views = [view_masks(image, view) for view in annotator_views(image)]
    total = PqCounts()
    for masks in views:
        total = total + fast_view_counts(kept, masks)
    base_counts.append(total)
    claimed.append(
        claim_file(
            detections_by_file[image.file_name],
            kept > 0,
            disk,
            views,
            RECTS,
            CLAIM_CONFIGS,
        )
    )
    if index == 1 or index % 10 == 0 or index == len(holdout):
        print(
            f"  {index}/{len(holdout)} {image.file_name} ({time.monotonic() - begin:.0f} s)",
            flush=True,
        )

base_pq = weighted_pq(base_counts)
print(f"base pooled PQ in sample: {base_pq:.4f} (expected {EXPECTED_BASE_PQ})")
if not PILOT and abs(base_pq - EXPECTED_BASE_PQ) > BASE_PQ_TOLERANCE:
    raise RuntimeError(
        f"The base PQ {base_pq:.4f} differs from {EXPECTED_BASE_PQ} by more than "
        f"{BASE_PQ_TOLERANCE}: the cache, the data or the pipeline is not the documented one."
    )

dedup = sum((file.dedup for file in claimed), DedupStats())
print(
    f"dedup: {dedup.detections} detections -> {dedup.merged} after the merge across windows "
    f"-> {dedup.kept} kept; {dedup.dropped} dropped, of which {dedup.dropped_cross_window} "
    f"were cross-window pairs the merge had left apart"
)
dates = holdout_dates(holdout)
default_claims = [file.claims[DEFAULT_CONFIG] for file in claimed]
calibrated = calibrate_crossfit(default_claims, dates)

setting_counts = {}
for setting in SETTINGS:
    setting_counts[setting] = [
        counts_for(
            file.claims[setting.config],
            setting.cut,
            scores=calibrated[position] if setting.calibrated else None,
            keep_unscored=setting.config.emit_unclaimed,
        )
        for position, file in enumerate(claimed)
    ]
detector_only = {
    "detector only, every detection (no U-Net)": [
        counts_for(file.detector_only) for file in claimed
    ],
    "detector only, raw>=0.5 (library default, untuned)": [
        counts_for(file.detector_only, 0.5) for file in claimed
    ],
}

## 5. Tables and the verdict

Each table shows the paired difference from the base over the same resampled files. The verdict is the cross-fit over every setting above plus the base, so a partition may keep the base. Read the `ADOPT` line and the last line. If it is `DO NOT ADOPT`, keep the current pipeline, however good a single row looks. The detector-only rows are a sanity check, not candidates.

Add one row to `docs/RESULTS.md` from this output and put the CSV in `results/`.

In [ ]:
rows = axis_rows(
    "base", base_counts, {"base (U-Net, flip TTA, mean>=0.8)": base_counts}
)
for axis, settings in AXES.items():
    rows += axis_rows(
        axis, base_counts, {setting: setting_counts[setting] for setting in settings}
    )
rows += axis_rows("sanity", base_counts, detector_only)
print(format_axis_rows(rows))

crossfit_settings = {"base": base_counts}
crossfit_settings.update({setting: setting_counts[setting] for setting in SETTINGS})
verdict_rows, adopted = crossfit_rows("all", base_counts, crossfit_settings, dates)
print(format_axis_rows(verdict_rows))
summary = crossfit_summary(base_counts, crossfit_settings, dates)
dedup_row = {
    "axis": "dedup",
    "setting": "dropped",
    "pooled_pq": "",
    "diff_vs_base": "",
    "diff_low": "",
    "diff_high": "",
    "fraction_better": "",
    "note": (
        f"{dedup.dropped} of {dedup.merged} merged detections dropped, "
        f"{dedup.dropped_cross_window} across windows"
    ),
}
write_axis_rows(OUTPUT_PATH, rows + verdict_rows + [dedup_row])
print(f"wrote {OUTPUT_PATH}")
print("This notebook read no test image, wrote no submission and did not submit.")
if PILOT:
    print("PILOT: a plumbing check on a few files. The line below is not a result.")
print(
    f"cross-fit adoption (axis_report rule): {'ADOPT' if adopted else 'DO NOT ADOPT'}"
)
print(f"settings chosen over the folds: {dict(summary.picks)}")
print(
    f"a submission without Nacho's OK needs diff > +{SUBMIT_GAIN} on the cross-fit number, "
    f"adopted >= {MIN_PARTITIONS_ADOPTED}/{summary.partitions} partitions and no pilot: "
    f"{'yes' if submit_without_asking(summary, pilot=PILOT) else 'no'}"
)
print(
    f"diff is measured against the base at its fixed setting, in sample ({base_pq:.4f}), "
    "not against the cross-fitted base (0.4210)"
)
print(verdict_line(summary))